In [1]:
import blocks
import torch
import torch.nn as nn

/home/cybertesla/Transformers/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
class GPT(nn.Module):
    """ One layer of GPT with attention and FFNN """

    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.attention = blocks.MaskedMultiHeadAttention(d_model, num_heads)
        self.ffnn = blocks.FFNN(d_model, hidden_dim)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dimension of x: (Batch_size, num_tokens, d_model)
        x = x + self.dropout(self.attention(self.norm1(x))) # Residual connection with Pre Normalization
        x = x + self.dropout(self.ffnn(self.norm2(x))) # Residual connection with Pre Normalization
        # Dimension: (Batch_size, num_tokens, d_model)
        return x

In [3]:
class GPTStack(nn.Module):
    """ A stack of GPT Layers """

    def __init__(self, vocab_size: int, d_model: int, num_blocks: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model     
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = blocks.PositionalEncoding()
        self.layers = nn.ModuleList([GPT(d_model, num_heads, hidden_dim, dropout_rate) for _ in range(num_blocks)])
        self.final_output_logits = nn.Linear(d_model, vocab_size)
        self.norm1 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dimension of tokens: (Batch_size, num_tokens)

        x = self.embedding(x)
        x = self.positional_encoding(x)
        # Dimension: (Batch_size, num_tokens, d_model)

        for layer in self.layers:
            x = layer(x)
        
        norm_output = self.norm1(x)

        logits = self.final_output_logits(norm_output)
        # Dimension: (Batch_size, num_tokens, vocab_size)

        return logits
        

In [4]:
batch_size = 3
num_tokens = 5
d_model = 8
num_heads = 2
head_dim = 4
vocab_size = 7
hidden_dim = 16
num_blocks = 6

In [6]:
dummy_input = torch.randint(low=0, high=vocab_size, size=(batch_size, num_tokens))
print(dummy_input.shape)
print("(batch_size, num_tokens)")

torch.Size([3, 5])
(batch_size, num_tokens)


In [8]:
model = GPTStack(vocab_size, d_model, num_blocks, num_heads, hidden_dim)
model

GPTStack(
  (embedding): Embedding(7, 8)
  (positional_encoding): PositionalEncoding()
  (layers): ModuleList(
    (0-5): 6 x GPT(
      (attention): MaskedMultiHeadAttention(
        (w_q): Linear(in_features=8, out_features=8, bias=False)
        (w_k): Linear(in_features=8, out_features=8, bias=False)
        (w_v): Linear(in_features=8, out_features=8, bias=False)
        (w_o): Linear(in_features=8, out_features=8, bias=False)
      )
      (ffnn): FFNN(
        (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
        (output_layer): Linear(in_features=16, out_features=8, bias=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_output_logits): Linear(in_features=8, out_features=7, bias=True)
  (norm1): LayerNorm((8,), eps=1e-05,

In [9]:
model_output = model(dummy_input)
print(model_output.shape)
print("(batch_size, num_tokens, vocab_size)")
print(model_output)

torch.Size([3, 5, 7])
(batch_size, num_tokens, vocab_size)
tensor([[[-0.5897, -0.6971,  0.2218, -0.1528, -1.0864,  0.6743, -0.5927],
         [-0.8656, -0.5433,  0.3015, -0.2178, -0.8708,  0.5520, -0.9136],
         [-1.0123,  0.4056,  0.5064, -0.8269, -0.1801,  0.6427, -1.5386],
         [-1.0772,  0.0750,  0.3966, -0.3635, -0.6787,  0.7763, -1.4147],
         [-0.7719, -1.0129,  0.1353, -0.0112, -0.9677,  0.5464, -0.5706]],

        [[-0.9254,  0.0505,  0.5901, -0.2521,  0.3177,  0.7003, -0.5248],
         [-1.2336,  0.2939,  0.3051,  0.1307, -0.0716, -0.0468, -0.5692],
         [-1.0885, -0.7130,  0.3316, -0.2355, -0.4775,  0.5903, -0.9577],
         [-1.1564,  0.8763,  0.4861, -0.7982,  0.1903,  0.8501, -1.3914],
         [-0.6882, -0.8491,  0.1791,  0.0353, -0.8228,  0.6161, -0.5519]],

        [[-1.0145,  0.0063,  0.5680, -0.3544,  0.2697,  0.8005, -0.6818],
         [-1.2442,  0.1634,  0.5895, -0.1143,  0.1709,  0.0354, -0.9633],
         [-1.1504,  0.5475,  0.3374, -0.9669,  0.